Cell 1 — Imports and paths

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from xgboost import XGBRegressor

BASE_DIR = Path(r"C:\Users\NickK\OneDrive\Υπολογιστής\github\kaggle_store_sales_forecasting")

ADVANCED_DIR = BASE_DIR / "data" / "advanced"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
SUBMISSION_DIR = BASE_DIR / "submissions"

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

Cell 2 — Load data

In [2]:
train = pd.read_csv(ADVANCED_DIR / "train_advanced.csv")
test = pd.read_csv(PROCESSED_DIR / "test_features.csv")

train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])

print(train.shape)
print(test.shape)

(3000888, 29)
(28512, 20)


Cell 3 — Create test lag/rolling features

In [3]:
latest_features = (
    train.sort_values(["store_nbr", "family", "date"])
    .groupby(["store_nbr", "family"])
    .tail(28)
)

test_lag_features = (
    latest_features
    .groupby(["store_nbr", "family"])
    .agg(
        sales_lag_7=("sales", lambda x: x.iloc[-7]),
        sales_lag_14=("sales", lambda x: x.iloc[-14]),
        sales_lag_28=("sales", lambda x: x.iloc[-28]),
        rolling_mean_7=("sales", lambda x: x.iloc[-7:].mean()),
        rolling_mean_14=("sales", lambda x: x.iloc[-14:].mean()),
        rolling_mean_28=("sales", lambda x: x.iloc[-28:].mean()),
        rolling_std_7=("sales", lambda x: x.iloc[-7:].std()),
        rolling_std_28=("sales", lambda x: x.iloc[-28:].std())
    )
    .reset_index()
)

test = test.merge(
    test_lag_features,
    on=["store_nbr", "family"],
    how="left"
)

lag_features = [
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28"
]

test[lag_features] = test[lag_features].fillna(0)

print(test.shape)

(28512, 28)


Cell 4 — Select features

In [4]:
target = "sales"

features = [
    "store_nbr",
    "family",
    "onpromotion",

    "year",
    "month",
    "day",
    "quarter",
    "week",
    "day_of_week",
    "is_weekend",

    "city",
    "state",
    "type",
    "cluster",

    "dcoilwtico",
    "is_payday",
    "is_national_holiday",
    "earthquake_period",

    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_28",

    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",

    "rolling_std_7",
    "rolling_std_28"
]

X = train[features]
y = np.log1p(train[target])
X_test = test[features]

Cell 5 — One-hot encode

In [5]:
X = pd.get_dummies(X, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)

print(X.shape)
print(X_test.shape)

(3000888, 94)
(28512, 94)


Cell 6 — Train final model

In [6]:
final_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

final_model.fit(X, y)

print("Final model trained.")

Final model trained.


Cell 7 — Predict test set

In [7]:
test_log_preds = final_model.predict(X_test)

test_preds = np.expm1(test_log_preds)
test_preds = np.maximum(test_preds, 0)

submission = pd.DataFrame({
    "id": test["id"],
    "sales": test_preds
})

submission.head()

,id,sales
0,3000888,4.055172
1,3000889,0.059201
2,3000890,4.649125
3,3000891,2104.148193
4,3000892,0.073674


Cell 8 — Save final submission

In [8]:
submission_path = SUBMISSION_DIR / "submission_final_xgboost_lags.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("Saved:", submission_path)
print(submission.shape)

Saved: C:\Users\NickK\OneDrive\Υπολογιστής\github\kaggle_store_sales_forecasting\submissions\submission_final_xgboost_lags.csv
(28512, 2)
